In [12]:
import yfinance as yf
import pandas as pd
import json

def explore_yfinance():
    # Let's test a US ETF and an Indian ETF
    test_tickers = ["SPY", "NIFTYBEES.NS"]
    
    for symbol in test_tickers:
        print(f"\n{'='*50}")
        print(f"Fetching Data for Ticker: {symbol}")
        print(f"{'='*50}")
        
        # 1. Initialize the Ticker object
        ticker = yf.Ticker(symbol)
        
        # 2. Fetch Metadata (Maps to `funds_master` and `amc_profiles`)
        print("\n--- 1. METADATA (funds_master table) ---")
        info = ticker.info
        
        # Cherry-picking the most relevant fields for our DB
        mapped_data = {
            "ticker_symbol": symbol,
            "fund_name": info.get('longName', 'N/A'),
            "category": info.get('category', 'N/A'),
            "fund_family": info.get('fundFamily', 'N/A'), # Maps to amc_name
            "total_assets": info.get('totalAssets', 'N/A'),
            "yield": info.get('yield', 'N/A'),
            "ytd_return": info.get('ytdReturn', 'N/A')
        }
        print(json.dumps(mapped_data, indent=4))
        
        # 3. Fetch Historical Prices (Maps to `nav_history`)
        print("\n--- 2. HISTORICAL DATA (nav_history table) ---")
        # period="5d" fetches the last 5 days of data
        hist = ticker.history(period="5d")
        
        # yfinance returns a Pandas DataFrame. We just want the Date and Close price.
        if not hist.empty:
            # Format it clearly to see how it will insert into our DB
            history_records = []
            for date, row in hist.iterrows():
                history_records.append({
                    "date": date.strftime('%Y-%m-%d'),
                    "close_price": round(row['Close'], 2)
                })
            
            print(f"Fetched {len(history_records)} days of history. Latest 3 days:")
            print(json.dumps(history_records[-4:], indent=4))
        else:
            print("No historical data found.")

# Run the exploration
explore_yfinance()


Fetching Data for Ticker: SPY

--- 1. METADATA (funds_master table) ---
{
    "ticker_symbol": "SPY",
    "fund_name": "State Street SPDR S&P 500 ETF Trust",
    "category": "Large Blend",
    "fund_family": "State Street Investment Management",
    "total_assets": 735060819968,
    "yield": 0.010299999,
    "ytd_return": 5.68005
}

--- 2. HISTORICAL DATA (nav_history table) ---
Fetched 5 days of history. Latest 3 days:
[
    {
        "date": "2026-05-05",
        "close_price": 723.77
    },
    {
        "date": "2026-05-06",
        "close_price": 733.83
    },
    {
        "date": "2026-05-07",
        "close_price": 731.58
    },
    {
        "date": "2026-05-08",
        "close_price": 737.62
    }
]

Fetching Data for Ticker: NIFTYBEES.NS

--- 1. METADATA (funds_master table) ---
{
    "ticker_symbol": "NIFTYBEES.NS",
    "fund_name": "Nippon India ETF Nifty 50 BeES",
    "category": "N/A",
    "fund_family": "N/A",
    "total_assets": "N/A",
    "yield": "N/A",
    "ytd_ret